Change tracks a bit, we gonna try running with foviate shrink

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import *

In [ ]:
from albumentations.augmentations.geometric.functional import bboxes_piecewise_affine
from mtrain.neg_mask.crops import get_region_crops, get_largest_bbox, padded_bbox
from mtrain.neg_mask.model.datasets.foviate_shrink import (
    do_foviate_shrink,
    get_foviate_remaps,
)
from mtrain.neg_mask.leveled_cropping import (
    load_crop_level_sample_from_directory,
    make_crop_level_pairs_v2,
)
from tqdm import tqdm
import albumentations as A


def _get_bbox_mask(bb, shape):
    zero = np.zeros(shape)
    zero[bb.y : bb.y2, bb.x : bb.x2] = 1
    return zero


def get_foviated_clean_crops(crop_level_path, full_image_size, crop_size, bbox_pad):
    pre_tfm = A.Compose(
        [
            A.Resize(full_image_size, full_image_size, cv2.INTER_AREA),
            A.PadIfNeeded(
                full_image_size, full_image_size, border_mode=cv2.BORDER_CONSTANT
            ),
        ],
        additional_targets={"bbox_mask": "mask"},
    )

    post_tfm = A.Compose(
        [
            A.Resize(crop_size, crop_size, cv2.INTER_AREA),
            A.PadIfNeeded(crop_size, crop_size, border_mode=cv2.BORDER_CONSTANT),
        ],
        additional_targets={"bbox_mask": "mask"},
    )

    for label in ["other", "trash"]:
        dirs = list((crop_level_path / label).glob("*"))
        for p in dirs:
            if (
                not p.is_dir()
                or not (p / "image.jpg").exists()
                or not (p / "source_dir" / "image.jpg").exists()
            ):
                continue

            try:
                sample = load_crop_level_sample_from_directory(p, full_image_size)
            except Exception as ex:
                print(f"WARN: failed in loading sample at {p.name} cause={ex}")
            img, mask = sample.full_image, sample.full_mask

            res = pre_tfm(
                image=img,
                mask=mask,
                bbox_mask=_get_bbox_mask(
                    padded_bbox(sample.bbox, 10, mask.shape), mask.shape
                ),
            )
            t_image, t_mask, t_bbox_mask = res["image"], res["mask"], res["bbox_mask"]
            t_bb = get_largest_bbox(t_bbox_mask)
            map_x, map_y = get_foviate_remaps(t_image.shape, t_bb, crop_size)
            re_img = cv2.remap(t_image, map_x, map_y, interpolation=cv2.INTER_LINEAR)
            re_mask = cv2.remap(t_mask, map_x, map_y, interpolation=cv2.INTER_LINEAR)

            res = post_tfm(image=re_img, mask=re_mask)
            re_img, re_mask = res["image"], res["mask"]

            yield (t_image, t_mask), (re_img, re_mask), label, p.stem

            # dest_dir = mkdir(root_dest_dir / label / p.name)
            # DiskImage.save(crop, dest_dir / "orig.jpg")
            # DiskBooleanMask.save(mask, dest_dir / "mask.png")

def save_foveated_crops(crop_level_dir, root_dest_dir, full_image_size, crop_size, bbox_pad):
    it = get_foviated_clean_crops(CROP_LEVEL_DIR, full_image_size, crop_size, bbox_pad)
    root_images_dir = mkdir(root_dest_dir / "train")
    root_masks_dir = mkdir(root_dest_dir / "masks")
    for item in tqdm(it):
        (img, mask), (re_img, re_mask), label, name = item
        fname = f"{label}_{name}"
        DiskImage.save(re_img, root_images_dir / f"{fname}.jpg")
        DiskBooleanMask.save(re_mask, root_masks_dir / f"{fname}.png")

In [ ]:
CROP_LEVEL_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level"
)
FOVEATED_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/foveated")

save_foveated_crops(CROP_LEVEL_DIR, FOVEATED_DIR, 1024, 224, 10)

In [ ]:
(img, mask), (re_img, re_mask) = next(it)
show([img, OV(img, mask), re_img, OV(re_img, re_mask)])

In [ ]:
from mtrain.example_dir import ExampleDir
from mtrain.seg import mapillary as mapi
from mtrain.example_dir.iterdir import get_dirs
WALLS_MAPILLARY = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/walls-mapillary")

RAW_DATA = WALLS_MAPILLARY / "raw_data"

dirs = list(get_dirs(RAW_DATA))

edir = ExampleDir(dirs[11], {}, {})
asts = edir.load_all_assets("md", "md")
mapi_pred = DiskBooleanMask.load(edir.mapi_mask_path())
wall_mask = mapi.get_mask(mapi_pred, mapi.Label.WALL)

wall_mask = wall_mask.astype(bool)
m2 = asts["m2"].astype(bool)


show([asts["image"], m2, wall_mask, OV(asts["image"], m2 & wall_mask)])

In [ ]:
from mtrain.neg_mask.crops import bbox_only_mask
wm = (wall_mask & m2).astype(np.uint8)
bboxes = list(get_region_crops(wm))

bb = bboxes[3]
res = bbox_only_mask(wm, bb, -1)
print(wm.shape, res.shape)
show([res, wm])

In [ ]:
# now create a foveated dataset for these masks.
# first goal is to create un-foveated dataset. I need to call this something
# L1 is what im going to call it lol

def make_l1_wall_masks(edir: ExampleDir):
    mapi_pred = DiskBooleanMask.load(edir.mapi_mask_path())
    wall_mask = mapi.get_mask(mapi_pred, mapi.Label.WALL)
    wall_mask = wall_mask.astype(bool)

    m2 = DiskBooleanMask.load(edir.trimmed_mask_path("md"))


    bboxes = list(get_region_crops(m2.astype(np.uint8)))
    res = []
    for bb in bboxes:
        bb_mask = bbox_only_mask(m2, bb, -1)
        if (bb_mask.astype(bool) & wall_mask.astype(bool)).sum() > 0:
            res.append(bb_mask)
    # wm = (m2 & wall_mask).astype(np.uint8)
    # bbs = list(get_region_crops(wm))

    # res = []
    # for bb in bbs:
    #     res.append(bbox_only_mask(wm, bb, -1))
    
    return edir.load_and_resize_image(edir.image_path), res


In [ ]:
def save_l1_masks(edir, image, masks, dest_dir):
    name = edir.d.name
    dest_dir = mkdir(dest_dir)

    images_dir = mkdir(dest_dir / "train")
    masks_dir = mkdir(dest_dir / "masks")

    for i, mask in enumerate(masks):
        fname = f"other_{name}_{i}"

        DiskImage.save(image, images_dir / f"{fname}.jpg")
        DiskBooleanMask.save(mask, masks_dir / f"{fname}.png")


In [ ]:
from tqdm import tqdm

WALLS_MAPILLARY_L1_DIR = WALLS_MAPILLARY / "L1"

dirs = list(get_dirs(RAW_DATA))
edirs = [ExampleDir(d, {}, {}) for d in dirs]

for edir in tqdm(edirs):
    try:
        image, masks = make_l1_wall_masks(edir)
        save_l1_masks(edir, image, masks, WALLS_MAPILLARY_L1_DIR)
    except Exception as ex:
        print(f"WARN: failed: {edir.d.name}, reason={ex}")


In [ ]:
from mtrain.example_dir.mapi_cons import MAPI_LABELS_TO_EXCLUDE
show_negmask_ds(WALLS_MAPILLARY_L1_DIR)

In [ ]:
# import numpy as np
# from pathlib import Path

# def stratified_sample_by_mask_size(path_size_pairs, n_per_bucket=300, n_buckets=10):
#     """
#     Stratified sampling of masks across decile buckets by mask size.
    
#     Args:
#         path_size_pairs: list of (Path, mask_size) tuples
#         n_per_bucket: number of samples per bucket (default 300)
#         n_buckets: number of decile buckets (default 10)
    
#     Returns:
#         sampled: list of (Path, mask_size) tuples
#     """
#     paths, sizes = zip(*path_size_pairs)
#     sizes = np.array(sizes)

#     # Compute decile bucket boundaries
#     percentiles = np.linspace(0, 100, n_buckets + 1)
#     boundaries = np.percentile(sizes, percentiles)

#     sampled = []
#     bucket_info = []

#     for i in range(n_buckets):
#         low, high = boundaries[i], boundaries[i + 1]

#         # Include upper bound only for the last bucket
#         if i < n_buckets - 1:
#             mask = (sizes >= low) & (sizes < high)
#         else:
#             mask = (sizes >= low) & (sizes <= high)

#         bucket_indices = np.where(mask)[0]

#         if len(bucket_indices) == 0:
#             continue

#         k = min(n_per_bucket, len(bucket_indices))
#         chosen = np.random.choice(bucket_indices, size=k, replace=False)
#         sampled.extend([(paths[j], sizes[j]) for j in chosen])
#         bucket_info.append({
#             "bucket": i + 1,
#             "range": (low, high),
#             "total": total_in_bucket,
#             "sampled": k,
#         })

#     return sampled

In [ ]:
def stratified_sample_by_mask_size(path_size_pairs, n_per_bucket=300, n_buckets=10):
    paths, sizes = zip(*path_size_pairs)
    sizes = np.array(sizes)

    percentiles = np.linspace(0, 100, n_buckets + 1)
    boundaries = np.percentile(sizes, percentiles)

    sampled = []
    bucket_info = []

    for i in range(n_buckets):
        low, high = boundaries[i], boundaries[i + 1]

        if i < n_buckets - 1:
            mask = (sizes >= low) & (sizes < high)
        else:
            mask = (sizes >= low) & (sizes <= high)

        bucket_indices = np.where(mask)[0]
        total_in_bucket = len(bucket_indices)

        k = min(n_per_bucket, total_in_bucket)
        chosen = np.random.choice(bucket_indices, size=k, replace=False) if k > 0 else []
        sampled.extend([(paths[j], sizes[j]) for j in chosen])

        bucket_info.append({
            "bucket": i + 1,
            "range": (low, high),
            "total": total_in_bucket,
            "sampled": k,
        })

    return sampled, bucket_info
# ```

# `bucket_info` will give you something like:
# ```
# [
#   {"bucket": 1, "range": (0, 120),    "total": 430, "sampled": 300},
#   {"bucket": 2, "range": (120, 540),  "total": 410, "sampled": 300},
#   ...
#   {"bucket": 9, "range": (3200, 8100),"total": 85,  "sampled": 85},  # fewer than 300
#   {"bucket": 10,"range": (8100, 51200),"total": 390, "sampled": 300},
# ]

In [ ]:
mask_paths = globL(WALLS_MAPILLARY_L1_DIR / "masks", "*.png")
path_size_pairs = [(path, DiskBooleanMask.load(path).sum()) for path in mask_paths]

In [ ]:
plt.hist([p[1] for p in path_size_pairs if p[1]< 1000])
plt.show()

In [ ]:
path_size_pairs = [p for p in path_size_pairs if p[1] > 30]

In [ ]:
sampled_path_size_pairs, bucket_info = stratified_sample_by_mask_size(path_size_pairs, 50, 100)

In [ ]:
bucket_info

In [ ]:
szs = [p[1] for p in path_size_pairs if p[1] < 250]
sampled_szs = [p[1] for p in sampled_path_size_pairs if p[1] < 250]

In [ ]:
len(sampled_path_size_pairs)

In [ ]:
import shutil
DEST_DIR = mkdir(WALLS_MAPILLARY / "L1-sampled")
mkdir(DEST_DIR / "train")
mkdir(DEST_DIR / "masks")

for p,sz in tqdm(sampled_path_size_pairs):
    name = p.stem
    src_image = WALLS_MAPILLARY_L1_DIR / "train" / f"{name}.jpg"
    src_mask = p

    dest_image = DEST_DIR / "train" / f"{name}.jpg"
    dest_mask = DEST_DIR / "masks" / f"{name}.png"

    shutil.copy(src_image, dest_image)
    shutil.copy(src_mask, dest_mask)

In [ ]:
show_negmask_ds(DEST_DIR)

In [ ]:
# foveate shrink now
from mtrain.neg_mask.model.datasets.foviate_shrink import get_foviated_image_and_mask

FOV_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/walls-mapillary/foveated")
mkdir(FOV_DIR / "train")
mkdir(FOV_DIR / "masks")

for img_path in tqdm(globL(DEST_DIR / "train", "*jpg")):
    mask_path = DEST_DIR / "masks" / f"{img_path.stem}.png"
    image = DiskImage.load(img_path)
    mask = DiskBooleanMask.load(mask_path)

    try:
        bbox = get_largest_bbox(mask)
    except:
        print(f"SKIP WARN: {img_path.name}")
        continue
    _, (re_img, re_mask) = get_foviated_image_and_mask(image, mask, bbox, 1024, 224, 10)

    fname = img_path.stem
    DiskImage.save(re_img, FOV_DIR / "train" / f"{fname}.jpg")
    DiskBooleanMask.save(re_mask, FOV_DIR / "masks" / f"{fname}.png")



In [ ]:
show_negmask_ds(FOV_DIR, 4  )

In [ ]:
from mtrain.neg_mask.model.datasets.copy_ds import copy_negmask_ds_to_ds
copy_negmask_ds_to_ds(FOV_DIR, "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/train", "mapillary-walls")

In [ ]:
image, masks = make_l1_wall_masks(edir)

In [ ]:
asts.keys()